In [0]:
%sql
select * from workspace.silver.silver_flights;

### **Parameters**

In [0]:
# # Catalog Name
# catalog = "workspace"
# # Key cols list
# key_cols = "['flight_id']"
# key_cols_list = eval("key_cols")

# # cdc column
# cdc_col = "modifiedDate"

# # back-dated refresh
# backdated_refresh = ""

# # source object
# source_object = "silver_flights"

# # source schema 
# source_schema = "silver"

# # target schema
# target_schema = "gold"

# # target object
# target_object = "DimFlights"

# # surrogate key column
# surrogate_key = "DimFlightsKey"

In [0]:
# # Catalog Name
# catalog = "workspace"
# # Key cols list
# key_cols = "['airport_id']"
# key_cols_list = eval("key_cols")

# # cdc column
# cdc_col = "modifiedDate"

# # back-dated refresh
# backdated_refresh = ""

# # source object
# source_object = "silver_airports"

# # source schema 
# source_schema = "silver"

# # target schema
# target_schema = "gold"

# # target object
# target_object = "DimAirports"

# # surrogate key column
# surrogate_key = "DimAirportsKey"

In [0]:
# Catalog Name
catalog = "workspace"
# Key cols list
key_cols = "['passenger_id']"
key_cols_list = eval("key_cols")

# cdc column
cdc_col = "modifiedDate"

# back-dated refresh
backdated_refresh = ""

# source object
source_object = "silver_passengers"

# source schema 
source_schema = "silver"

# target schema
target_schema = "gold"

# target object
target_object = "DimPassengers"

# surrogate key column
surrogate_key = "DimPassengersKey"

### **INCREMENTAL DATA INGESTION**

### last load date

In [0]:
# No back-dated refresh
if len(backdated_refresh) == 0:
    if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
        last_load = spark.sql(f"SELECT max({cdc_col}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
    else:
        last_load = "1900-01-01 00:00:00"
else:
    last_load = backdated_refresh     

In [0]:
last_load

In [0]:
df_src = spark.sql(f"SELECT * FROM {source_schema}.{source_object} where {cdc_col} >='{last_load}'")

In [0]:
df_src.display()

### **OLD VS NEW RECORDS**

In [0]:
import ast

# Convert key columns into an actual Python list
if isinstance(key_cols_list, str):
    key_cols_list = ast.literal_eval(key_cols_list)

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    # Key columns string
    key_cols_string_incremental = ', '.join(key_cols_list)

    df_trg = spark.sql(
        f"""SELECT {key_cols_string_incremental},
                   {surrogate_key},
                   create_date,
                   update_date
            FROM {catalog}.{target_schema}.{target_object}"""
    )

else:
    key_cols_string_init = ', '.join(
        [f"'' AS {i}" for i in key_cols_list]
    )

    df_trg = spark.sql(
        f"""SELECT {key_cols_string_init},
                   cast('0' AS INT) AS {surrogate_key},
                   cast('1900-01-01 00:00:00' AS timestamp) AS create_date,
                   cast('1900-01-01 00:00:00' AS timestamp) AS update_date where 1=0"""
    )

In [0]:
df_trg.display()

In [0]:
df_trg.display()

### **join condition**

In [0]:
join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_cols_list])

In [0]:
df_src = spark.sql(
    f"SELECT * FROM {catalog}.{source_schema}.{source_object}"
)

df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

df_join = spark.sql(f"""
    SELECT src.*, 
           trg.{surrogate_key},
           trg.create_date,
           trg.update_date
    FROM src
    LEFT JOIN trg
    ON {join_condition}
""")

In [0]:
df_join.display()

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
# old records
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
# new records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())

In [0]:
df_old.display()
df_new.display()

### **Enriching dfs**

### **Preparing df_old**

In [0]:
df_old_enr = df_old.withColumn('update_date',current_timestamp())

In [0]:
df_old_enr.display()

### **Preparing df_new**

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
  max_surrogate_key = spark.sql(f"""select max({surrogate_key})
          from {catalog}.{target_schema}.{target_object}""").collect()[0][0]
  
  df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
      .withColumn('create_date',current_timestamp())\
      .withColumn('update_date',current_timestamp())

else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
      .withColumn('create_date',current_timestamp())\
      .withColumn('update_date',current_timestamp())

In [0]:
max_surrogate_key

In [0]:
df_new_enr.display()

In [0]:
spark.sql("""select max(flight_id) as max_surrogate_key
          from workspace.silver.silver_flights""").collect()[0][0]

In [0]:
df_old_enr.display()

### **Union old and new records**

In [0]:
df_union = df_old.unionByName(df_new_enr)

In [0]:
df_union.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
df_union.write.format("delta")\
    .mode("append")\
    .saveAsTable(f"{catalog}.{target_schema}.{target_object}")      

In [0]:
key_cols_string_incremental = ', '.join(key_cols_list)

### **Upserting stepwise**

In [0]:
# Define target table
target_table = f"{catalog}.{target_schema}.{target_object}"

# Read target table
df_trg = spark.table(target_table)

# Define backup table
backup_table = f"{catalog}.{target_schema}.{target_object}_backup"

# Create backup only if it does not already exist
df_trg.write.format("delta") \
    .mode("ignore") \
    .saveAsTable(backup_table)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

window_spec = Window.partitionBy(*key_cols_list) \
    .orderBy(desc(cdc_col), desc("update_date"))

df_trg_clean = df_trg.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")

df_trg_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc, current_timestamp

# 1. Deduplicate source using business key
window_spec = Window.partitionBy(*key_cols_list) \
    .orderBy(desc(cdc_col))

df_union = df_union.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")


# 2. Check for duplicate surrogate keys in source
duplicate_keys = df_union.groupBy(surrogate_key) \
    .count() \
    .filter(col("count") > 1)

if duplicate_keys.limit(1).count() > 0:
    raise ValueError("Duplicate surrogate keys found in source!")


# 3. Check if target exists
if spark.catalog.tableExists(target_table):

    dlt_obj = DeltaTable.forName(spark, target_table)

    # Update columns except create_date and update_date
    update_set = {
        c: f"src.`{c}`"
        for c in df_union.columns
        if c not in ["create_date", "update_date"]
    }

    update_set["update_date"] = "current_timestamp()"

    # Insert values
    insert_set = {
        c: f"src.`{c}`"
        for c in df_union.columns
        if c not in ["create_date", "update_date"]
    }

    insert_set["create_date"] = "current_timestamp()"
    insert_set["update_date"] = "current_timestamp()"

    # 4. Merge
    dlt_obj.alias("trg").merge(
        df_union.alias("src"),
        f"trg.{surrogate_key} = src.{surrogate_key}"
    ) \
    .whenMatchedUpdate(
        condition=f"src.{cdc_col} > trg.{cdc_col}",
        set=update_set
    ) \
    .whenNotMatchedInsert(
        values=insert_set
    ) \
    .execute()

else:
    # 5. Create target if it doesn't exist
    df_union.withColumn(
        "create_date", current_timestamp()
    ).withColumn(
        "update_date", current_timestamp()
    ).write.format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

In [0]:
%sql 
select * from workspace.gold.dimpassengers where passenger_id = 'P0049'